In [ ]:
from PIL import Image, ImageOps
import numpy as np
import cv2
import os
import cupy as cp
import matplotlib.pyplot as plt
from ultralytics import YOLO
import glob

In [ ]:
#bboxの学習

model = YOLO('/home/src/yolo/yolo12n.pt')

file = "0620_bbox"

results = model.train(
    data=f'/home/YOLO/{file}/datasets/config.yaml', 
    epochs=1000, 
    imgsz=640, 
    project=f"/home/YOLO/{file}/datasets", 
    device='mps'   
)

In [ ]:
# segmantationの学習
model = YOLO('/home/src/yolo/yolo11n-seg.pt')

file = "-327_seg"

results = model.train(
    data=f'/home/YOLO/{file}/datasets/config.yaml', 
    epochs=1000, 
    imgsz=640, 
    project=f"/home/YOLO/{file}/datasets", 
    device='mps'   
)

In [ ]:
# segmantationの学習
model = YOLO('/home/src/yolo/yolo11x-seg.pt')

file = "-327_seg"

results = model.train(
    data=f'/home/YOLO/{file}/datasets/config.yaml', 
    epochs=1000, 
    imgsz=640, 
    project=f"/home/YOLO/{file}/datasets", 
    device='mps'   
)


In [ ]:
# segmantationの評価
# model = YOLO("/home/YOLO/-327_seg/datasets/train2/weights/best.pt")  
# model = YOLO("/home/YOLO/hukusuu_train/datasets/train7/weights/best.pt")  
model = YOLO("/home/YOLO/1008_bbox/datasets/train2/weights/best.pt")

# 評価データセットでモデルを評価
metrics = model.val(workers=0)

# 結果を見やすく表示
print("=== Model Evaluation Metrics ===")
print(f"mAP@50: {metrics.box.map50:.3f}")       # mAP@50
print(f"mAP@50-95: {metrics.box.map:.3f}")      # mAP@50-95（IoU 0.5~0.95の平均）
print(f"Precision: {metrics.box.mp:.3f}")       # 平均 Precision
print(f"Recall: {metrics.box.mr:.3f}")          # 平均 Recall
print(f"Number of Classes: {metrics.box.nc}")   # クラス数
print(f"Processing Speed: {metrics.speed}")     # 処理速度情報
print("================================")


In [ ]:
#segmentationのテスト
import numpy as np
import cv2

# 640x480の黒画像を作成
black_image = np.zeros((1536, 2048), dtype=np.uint8)

# YOLOモデルの読み込みと推論
model = YOLO('/home/YOLO/-327_seg/datasets/train2/weights/best.pt')
# model = YOLO('/home/YOLO/-327_seg/datasets/train/weights/best.pt')

path = "/home/YOLO/-327_seg/datasets/images/test/IMG_1902.JPEG"
results = model(path)  # results list

# 結果のマスク座標を取得し、黒画像に白で描写
for i, r in enumerate(results):
    # print(f"Result {i}:")
    # for mask in r.masks.xy:
    #     print(mask)  # マスク座標をプリント
    #     # 座標を整数に変換
    #     points = np.array(mask, dtype=np.int32)
    #     # ポリゴンを白で描写
    #     cv2.fillPoly(black_image, [points], 255)
    
    r.show(boxes=False,masks=True)

# 結果を保存
# cv2.imwrite("masked_image.jpg", black_image)

In [ ]:
#save crop shiitake test

# model = YOLO('/home/YOLO/hukusuu_train/datasets/train7/weights/best.pt')
model = YOLO('/home/YOLO/0917_bbox/datasets/train2/weights/best.pt')

path = "/home/data/maesyori_img/collage_4.jpg"
# metrics = model.val()
# print("=== Model Evaluation Metrics ===")
# print(f"mAP@50: {metrics.box.map50:.3f}")       # mAP@50
# print(f"mAP@50-95: {metrics.box.map:.3f}")      # mAP@50-95
# print(f"Precision: {metrics.box.mp:.3f}")       # 平均 Precision
# print(f"Recall: {metrics.box.mr:.3f}")          # 平均 Recall
# print(f"Number of Classes: {metrics.box.nc}")   # クラス数
# print(f"Processing Speed: {metrics.speed}")     # 処理速度情報
# print("================================")

#save_crop:save_dir/cls/file_nameに保存
# Visualize the results
# for i, r in enumerate(results):
    # r.show()

In [ ]:
#検出テスト
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

# モデルの読み込み
model = YOLO('/home/YOLO/hukusuu_train/datasets/train7/weights/best.pt')

# 画像の読み込み
image_path = "/home/data/maesyori_img/collage_4.jpg"
image = cv2.imread(image_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# 画像サイズ取得
H, W, _ = image.shape

# 縦4×横6のグリッドのサイズ
rows, cols = 4, 6
cell_H, cell_W = H // rows, W // cols

# 物体検出
results = model(image_path)

# 各マスに含まれるbboxを整理（マスの順番に並ぶように）
grid_bboxes = [[[] for _ in range(cols)] for _ in range(rows)]

for box in results[0].boxes.xyxy:
    start_x, start_y, end_x, end_y = map(int, box)
    center_x, center_y = (start_x + end_x) // 2, (start_y + end_y) // 2  # 中心座標

    # どのマスに入るか計算
    grid_row = min(center_y // cell_H, rows - 1)
    grid_col = min(center_x // cell_W, cols - 1)

    grid_bboxes[grid_row][grid_col].append(box)

# 可視化
plt.figure(figsize=(12, 6))
plt.imshow(image)
colors = ['red', 'blue', 'green', 'orange']  # 4色をローテーション

# マス目ごとに整理されたbboxを描画
index = 1
for row in range(rows):
    for col in range(cols):
        for bbox in grid_bboxes[row][col]:
            start_x, start_y, end_x, end_y = map(int, bbox)
            color = colors[(index - 1) % 4]  # 4色ローテーション
            plt.gca().add_patch(plt.Rectangle((start_x, start_y), end_x - start_x, end_y - start_y, 
                                              linewidth=2, edgecolor=color, facecolor='none'))
            plt.text(start_x, start_y - 5, str(index), color=color, fontsize=12, fontweight='bold')
            index += 1

plt.axis('off')
plt.show()


In [ ]:
#複数シイタケ画像から個々の切り取り
import cv2
import numpy as np
import os
glob
from ultralytics import YOLO

# モデルの読み込み
model = YOLO('/home/YOLO/hukusuu_train/datasets/train7/weights/best.pt')

# 画像フォルダのパス
image_folder = "/home/data/maesyori_img/"
output_folder = "img/"
os.makedirs(output_folder, exist_ok=True)

# すべての画像を処理
def process_image(image_path):
    results = model(image_path)  # 結果を取得

    # 画像の読み込み
    orig_img = results[0].orig_img
    img_h, img_w, _ = orig_img.shape

    # 4×6のマスに分割
    rows, cols = 4, 6
    cell_h, cell_w = img_h // rows, img_w // cols

    # マス目ごとの bbox を格納するリスト
    cell_bboxes = [[[] for _ in range(cols)] for _ in range(rows)]

    # bbox を対応するマス目に格納
    for i, box in enumerate(results[0].boxes.xyxy):
        start_x, start_y, end_x, end_y = map(int, box)
        center_x, center_y = (start_x + end_x) // 2, (start_y + end_y) // 2
        
        row_idx = min(center_y // cell_h, rows - 1)
        col_idx = min(center_x // cell_w, cols - 1)
        
        cell_bboxes[row_idx][col_idx].append((start_x, start_y, end_x, end_y))

    # 出力フォルダを作成
    base_name = os.path.splitext(os.path.basename(image_path))[0]
    image_output_folder = os.path.join(output_folder, base_name)
    os.makedirs(image_output_folder, exist_ok=True)

    # マス目順に bbox を保存
    for row in range(rows):
        for col in range(cols):
            for idx, (sx, sy, ex, ey) in enumerate(cell_bboxes[row][col]):
                clip_img = orig_img[sy:ey, sx:ex]
                clip_filename = os.path.join(image_output_folder, f"clip_{row+1}_{col+1}.jpg")
                cv2.imwrite(clip_filename, clip_img)
                # print(f"保存: {clip_filename}")

# フォルダ内のすべての画像を処理
image_paths = glob.glob(os.path.join(image_folder, "*.jpg"))
for image_path in image_paths:
    process_image(image_path)


ケースの複数画像からマスク画像を得るため

In [ ]:
#bboxの学習

model = YOLO('/home/src/yolo/yolo12n.pt')

file = "0708_maesyori"

results = model.train(
    data=f'/home/YOLO/{file}/datasets/config.yaml', 
    epochs=1000, 
    imgsz=640, 
    project=f"/home/YOLO/{file}/datasets", 
    device='mps'   
)

In [ ]:
# segmantationの学習
model = YOLO('/home/src/yolo/yolo11n-seg.pt')

file = "0728_segmet"

results = model.train(
    data=f'/home/YOLO/{file}/datasets/config.yaml', 
    epochs=1000, 
    imgsz=640, 
    project=f"/home/YOLO/{file}/datasets", 
    device='mps'   
)

In [2]:
from ultralytics import YOLO

# 学習済みのモデルを読み込む
model = YOLO('/home/YOLO/0728_segmet/datasets/train2/weights/best.pt')

file = "0728_segmet"

# モデルの精度を検証データで評価する
# data引数にはdata.yamlファイルのパスを指定します
metrics = model.val(data=f'/home/YOLO/{file}/datasets/config.yaml')

# 主要な評価指標を表示
print("mAP50-95:", metrics.box.map)    # mAP@.50-.95 (主要な指標)
print("mAP50:", metrics.box.map50)     # mAP@.50
print("mAP75:", metrics.box.map75)     # mAP@.75

# 各クラスごとのmAP50-95
print("mAPs per class:", metrics.box.maps)

Ultralytics 8.3.63 🚀 Python-3.11.9 torch-2.6.0+cu124 CPU (Intel Core(TM) i9-14900K)


YOLO11n-seg summary (fused): 265 layers, 2,834,763 parameters, 0 gradients, 10.2 GFLOPs


val: Scanning /home/YOLO/0728_segmet/datasets/labels/val.cache... 50 images, 0 backgrounds, 2 corrupt: 100%|██████████| 50/50 [00:00<?, ?it/s]

val: WARNING ⚠️ /home/YOLO/0728_segmet/datasets/images/val/IMG_2657_shiitake_23.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [       1.06]
val: WARNING ⚠️ /home/YOLO/0728_segmet/datasets/images/val/IMG_2666_shiitake_28.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.0222]



/usr/local/lib/python3.11/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:01<00:00,  1.51it/s]


                   all         48         48      0.999          1      0.995      0.994      0.999          1      0.995      0.995
Speed: 0.7ms preprocess, 36.6ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to /home/yolov12/runs/segment/val7
mAP50-95: 0.9938789610389611
mAP50: 0.995
mAP75: 0.995
mAPs per class: [    0.99388]
